<a href="https://colab.research.google.com/github/ZSNonSKY/Lora_Trainer_Anima/blob/main/Lora_Trainer_Anima.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🌟 Anima Lora Trainer

A fork of [hollowstrawberry](https://github.com/hollowstrawberry)'s [XL Lora Trainer](https://github.com/hollowstrawberry/kohya-colab), rebuilt to train Loras for [Anima](https://huggingface.co/circlestone-labs/Anima) by circlestone-labs — a 2B parameter Diffusion Transformer built on NVIDIA's Cosmos-Predict2, using a Qwen3-0.6B text encoder and the Qwen-Image VAE.

Unlike the original notebook this fork trains directly against [kohya-ss/sd-scripts](https://github.com/kohya-ss/sd-scripts)'s native `anima_train_network.py`, rather than through a wrapper backend, since Anima's training needs (Rectified Flow, no conv layers, an optional LLM adapter) don't map onto the SDXL settings that trainer was built around.

Colab's free T4 GPU (15GB VRAM) is enough for this. Colab Pro with a better GPU will train faster and allow bigger batches.

This colab is possible thanks to open source code from talented people:
* [kohya-ss](https://github.com/kohya-ss/sd-scripts) for sd-scripts and its native Anima support
* [circlestone-labs](https://huggingface.co/circlestone-labs) for Anima
* [hollowstrawberry](https://github.com/hollowstrawberry/kohya-colab) for the original XL Lora Trainer this fork is based on
* [derrian-distro](https://github.com/derrian-distro/LoRA_Easy_Training_scripts_Backend/), [Linaqruf](https://github.com/Linaqruf/kohya-trainer) and [Jelosus2](https://github.com/Jelosus2/LoRA_Easy_Training_Colab), whose earlier work shaped the original notebook this one descends from

### ⭕ Disclaimer
The purpose of this document is to research bleeding-edge technologies in the field of machine learning.
Please read and follow the [Google Colab guidelines](https://research.google.com/colaboratory/faq.html) and its [Terms of Service](https://research.google.com/colaboratory/tos_v3.html).

| |GitHub|
|:--|:-:|
| 🏠 **Homepage** | [![GitHub](https://raw.githubusercontent.com/hollowstrawberry/kohya-colab/main/assets/github.svg)](https://github.com/ZSNonSKY/Lora_Trainer_Anima) |
| 🌟 **Anima Lora Trainer** | [![GitHub](https://raw.githubusercontent.com/hollowstrawberry/kohya-colab/main/assets/github.svg)](https://github.com/ZSNonSKY/Lora_Trainer_Anima/blob/main/Anima_Lora_Trainer.ipynb) |
| 🌟 **Original XL Lora Trainer** | [![GitHub](https://raw.githubusercontent.com/hollowstrawberry/kohya-colab/main/assets/github.svg)](https://github.com/hollowstrawberry/kohya-colab/blob/main/Lora_Trainer_XL.ipynb) |


In [ ]:
import os
import re
import sys
import codecs
import subprocess
import toml
from time import time
from IPython.display import Markdown, display
import warnings

warnings.filterwarnings("ignore")

# These carry information from past executions
if "model_url" in globals():
  old_model_url = model_url
else:
  old_model_url = None
if "dependencies_installed" not in globals():
  dependencies_installed = False
if "model_file" not in globals():
  model_file = None

# These may be set by other cells, some are legacy
if "custom_dataset" not in globals():
  custom_dataset = None
if "override_dataset_config_file" not in globals():
  override_dataset_config_file = None
if "continue_from_lora" not in globals():
  continue_from_lora = ""
if "override_config_file" not in globals():
  override_config_file = None

SDSCRIPTS_TAG = "v0.11.1"  # kohya-ss/sd-scripts official release, pinned for reproducibility and stability
LOWRAM = True
LOAD_TRUNCATED_IMAGES = True
BETTER_EPOCH_NAMES = True
SIGMOID_SCALE = 1.3  # Anima author-recommended value for sigmoid/shift/flux_shift timestep sampling, set internally rather than exposed


#@title ## 🚩 Start Here

#@markdown ### ▶️ Setup
#@markdown Your project name will be the same as the folder containing your images. Spaces aren't allowed.
project_name = "" #@param {type:"string"}
project_name = project_name.strip()
#@markdown The folder structure doesn't matter and is purely for comfort. Make sure to always pick the same one. I like organizing by project.
folder_structure = "Organize by category (MyDrive/lora_training/datasets/project_name)" #@param ["Organize by category (MyDrive/lora_training/datasets/project_name)", "Organize by project (MyDrive/Loras/project_name/dataset)"]
#@markdown Choose the Anima diffusion model (DiT) that will be downloaded and used for training. Leave the box below empty to use the official **Anima Base v1.0** checkpoint.<p>
#@markdown You can paste a direct link to any custom Anima finetune's `.safetensors` file here instead (Hugging Face "resolve" links and civitai links both work), or a path in your Google Drive starting with `/content/drive/MyDrive`. This must be a single-file checkpoint like the ones published under `split_files/diffusion_models` on Hugging Face — a Diffusers-format repo (like `Anima-Base-v1.0-Diffusers`) can't be used here, since this trainer needs one file, not a diffusers folder.
optional_custom_training_model = "" #@param {type:"string"}
#@markdown Use wandb if you want to visualize the progress of your training over time.
wandb_key = "" #@param {type:"string"}

default_model_url = "https://huggingface.co/circlestone-labs/Anima/resolve/main/split_files/diffusion_models/anima-base-v1.0.safetensors"
model_url = (optional_custom_training_model or default_model_url).strip()

# The Qwen3 text encoder and Qwen-Image VAE stay the same no matter which diffusion model above is chosen
qwen3_url = "https://huggingface.co/circlestone-labs/Anima/resolve/main/split_files/text_encoders/qwen_3_06b_base.safetensors"
qwen3_file = "/content/qwen_3_06b_base.safetensors"
vae_url = "https://huggingface.co/circlestone-labs/Anima/resolve/main/split_files/vae/qwen_image_vae.safetensors"
vae_file = "/content/qwen_image_vae.safetensors"

#@markdown ### ▶️ Processing
resolution = 1024 #@param {type:"slider", min:512, max:1536, step:64}
caption_extension = ".txt" #@param [".txt", ".caption"]
#@markdown Shuffling anime tags in place improves learning and prompting. An activation tag goes at the start of every text file and will not be shuffled.<p>
shuffle_tags = True #@param {type:"boolean"}
shuffle_caption = shuffle_tags
activation_tags = "1" #@param [0,1,2,3]
keep_tokens = int(activation_tags)

#@markdown ### ▶️ Steps <p>
#@markdown Your images will repeat this number of times during training. I recommend that your images multiplied by their repeats is around 100, or 1 repeat with more than 100 images.
num_repeats = 2 #@param {type:"number"}
#@markdown Choose how long you want to train for. A good starting point is around 10 epochs or around 1500 steps — Anima trains slower per-step than SDXL, but also tends to learn quicker (fewer steps needed), which roughly balances out; this range is aimed at comfortably fitting inside the free T4's 5 hour/day quota.<p>
#@markdown One epoch is a number of steps equal to: your number of images multiplied by their repeats, divided by batch size. <p>
preferred_unit = "Epochs" #@param ["Epochs", "Steps"]
how_many = 10 #@param {type:"number"}
max_train_epochs = how_many if preferred_unit == "Epochs" else None
max_train_steps = how_many if preferred_unit == "Steps" else None
#@markdown Saving more epochs will let you compare your Lora's progress better.
save_every_n_epochs = 1 #@param {type:"number"}
keep_only_last_n_epochs = 10 #@param {type:"number"}
if not save_every_n_epochs:
  save_every_n_epochs = max_train_epochs
if not keep_only_last_n_epochs:
  keep_only_last_n_epochs = max_train_epochs

#@markdown ### ▶️ Learning
#@markdown The learning rate is the most important setting for your results. If your Lora produces garbled or broken images, lower the dit and text encoder rates, e.g. to 1e-4 and 1e-5 respectively, or even lower. <p>
#@markdown If your style isn't baking in strongly enough, try raising unet_lr a bit. If you're training a style and seeing concept dilution (it bleeding into or fighting with the base model's own concepts), try setting the text encoder to 0 instead of just lowering it -- though note it's disabled by default anyway below, since cache_text_encoder_outputs defaults to on.
unet_lr = 2e-4 #@param {type:"number"}
text_encoder_lr = 2e-5 #@param {type:"number"}
#@markdown The scheduler is the algorithm that guides the learning rate. If you're not sure, pick `constant` and ignore the number. `cosine` is recommended here for Anima.
lr_scheduler = "cosine" #@param ["constant", "cosine", "cosine_with_restarts", "constant_with_warmup", "linear", "polynomial"]
lr_scheduler_number = 3 #@param {type:"number"}
#@markdown Steps spent "warming up" the learning rate during training for efficiency. I recommend leaving it at 5%.
lr_warmup_ratio = 0.05 #@param {type:"slider", min:0.0, max:0.2, step:0.01}
lr_warmup_steps = 0
#@markdown Anima is trained with Rectified Flow instead of the epsilon-prediction SDXL uses, so a couple of classic SDXL settings don't do anything here and have been removed from this trainer entirely: **min_snr_gamma** and **ip_noise_gamma** are both silently ignored by kohya-ss/sd-scripts' Anima training code. Anima's own equivalents (how timesteps are sampled/weighted) are kept fixed at their recommended defaults in the background rather than exposed here, since they're easy to misjudge coming from an SDXL mindset and the defaults are already the right call for almost everyone.
timestep_sampling = "sigmoid"
discrete_flow_shift = 1.0
weighting_scheme = "uniform"

#@markdown ### ▶️ Structure
#@markdown LoRA is the classic type, using Anima's native `networks.lora_anima` module. LoHa (Hadamard product) is also natively supported for Anima by kohya-ss/sd-scripts as `networks.loha`, and can sometimes capture more detail at the same rank -- but in testing it runs about 40% slower than LoRA on a T4, so LoRA is the default here given the free tier's time limits.
lora_type = "LoRA" #@param ["LoRA", "LoHa"]

#@markdown Below are the recommended starting values for Anima:

#@markdown | type | network_dim | network_alpha |
#@markdown | :---: | :---: | :---: |
#@markdown | LoRA | 32 | 32 |
#@markdown | LoHa | 16 | 16 |

#@markdown Anima is a Diffusion Transformer with no convolutional layers, so unlike SDXL/LoCon there's no separate "conv rank" here — dim/alpha apply to every trained module. More dim means a larger, more expressive Lora, but isn't always better.
network_dim = 32 #@param {type:"slider", min:1, max:64, step:1}
network_alpha = 32 #@param {type:"slider", min:1, max:64, step:1}

#@markdown ### ▶️ LLM Adapter
#@markdown Anima bridges its Qwen3 text encoder to the DiT through a small LLM Adapter. **circlestone-labs recommend leaving this at 0** so the adapter stays frozen and your Lora doesn't distort how prompts are interpreted. Only raise it above 0 if you specifically want to retrain that bridge, and know what you're doing — this enables training the adapter and gives it its own learning rate.
llm_adapter_lr = 0 #@param {type:"number"}

network_module = "networks.loha" if lora_type.lower() == "loha" else "networks.lora_anima"
network_args = []
if llm_adapter_lr and float(llm_adapter_lr) > 0:
  network_args.append("train_llm_adapter=True")
  network_args.append(f"network_reg_lrs=.*llm_adapter.*={llm_adapter_lr}")
network_args = network_args or None

#@markdown ### ▶️ Training
#@markdown Adjust these parameters depending on your colab configuration.
#@markdown
#@markdown Higher batch size is often faster but uses more memory. If you hit a CUDA out-of-memory error, lower this instead of raising GAS below.
train_batch_size = 2 #@param {type:"slider", min:1, max:16, step:1}
#@markdown Gradient Accumulation Steps: instead of updating every batch, accumulate gradients over this many batches first. This gives you a larger *effective* batch size (`train_batch_size × GAS`) without the VRAM cost of actually loading that many images at once -- useful if train_batch_size above is already maxing out your VRAM but you want the stability of a bigger effective batch.
gradient_accumulation_steps = 1 #@param {type:"slider", min:1, max:8, step:1}
#@markdown `torch` uses PyTorch's built-in scaled_dot_product_attention (SDPA), which automatically picks the fastest attention kernel your GPU supports (on a T4 that's the memory-efficient backend, since flash-attention-2 needs Ampere or newer) — no extra install needed. `xformers` is a separate library that can be a little faster on some GPUs, but needs its own install step (already handled in install_trainer() below) with a version matched to torch. I have found no substantial difference between the two in practice.
cross_attention = "torch" #@param ["torch", "xformers"]
#@markdown Use `full fp16` for lowest memory usage. **The free Colab T4 GPU (15GB VRAM) does not support bf16** — stick to `full fp16` on a T4. Also note: **`full fp16` doesn't play well with Prodigy/ProdigyPlusScheduleFree** (their internal learning-rate math needs more precision than fp16 weights give it and can silently fail to learn) -- if you're using one of those optimizers, use `mixed fp16` on a T4, or a bf16 option if you've upgraded to a better GPU.<p>
#@markdown The Lora will be trained with the selected precision, but will always be saved in fp16 format for compatibility reasons.
precision = "full fp16" #@param ["float", "full fp16", "full bf16", "mixed fp16", "mixed bf16"]
#@markdown Caching latents to drive will add a small file next to each image but will use considerably less memory.
cache_latents = True #@param {type:"boolean"}
cache_latents_to_drive = True #@param {type:"boolean"}
#@markdown Enabling this turns off shuffle_tags and text encoder training, but in testing it also runs about 20% faster than a normal Lora run -- worth it for most people on a T4, which is why it's on by default. Turn it off if you specifically need the text encoder trained (e.g. teaching new vocabulary) or need tag shuffling.
cache_text_encoder_outputs = True #@param {type:"boolean"}

mixed_precision = "no"
if "fp16" in precision:
  mixed_precision = "fp16"
elif "bf16" in precision:
  mixed_precision = "bf16"
full_precision = "full" in precision

#@markdown ### ▶️ Memory optimization (Anima-specific)
#@markdown If you run out of VRAM, raise this to offload some transformer blocks to CPU RAM during the forward/backward pass. Costs speed, saves VRAM. 0 = disabled — with the settings above this usually isn't needed for a Lora on a T4.
blocks_to_swap = 0 #@param {type:"slider", min:0, max:34, step:1}
#@markdown Splits the VAE's spatial processing into chunks to use less VRAM, at some cost to speed. Mainly useful at higher resolutions. 0 = disabled.
vae_chunk_size = 0 #@param {type:"slider", min:0, max:256, step:16}

#@markdown ### ▶️ Advanced
#@markdown The optimizer is the algorithm used for training. AdamW8bit is the default and works great, while Prodigy manages learning rate automatically and may have several advantages such as training faster due to needing less steps as well as working better for small datasets. ProdigyPlusScheduleFree is a community optimizer (Prodigy's auto learning rate + schedule-free averaging in one) that many people report even better results with than plain Prodigy — it needs `prodigy-plus-schedule-free` and `schedulefree`, both already included by sd-scripts' own requirements.txt, so nothing extra to install.
optimizer = "AdamW8bit" #@param ["AdamW8bit", "Prodigy", "ProdigyPlusScheduleFree", "DAdaptation", "DadaptAdam", "DadaptLion", "AdamW", "Lion", "SGDNesterov", "SGDNesterov8bit", "AdaFactor"]
#@markdown Recommended args for AdamW8bit: `weight_decay=0.1 betas=[0.9,0.99]`<p>
#@markdown Recommended args for Prodigy: `decouple=True weight_decay=0.01 betas=[0.9,0.999] d_coef=2 use_bias_correction=True safeguard_warmup=True`<p>
#@markdown Recommended args for ProdigyPlusScheduleFree: `weight_decay=0.1`. Because it's schedule-free, it doesn't need (or benefit from) an external lr_scheduler — `constant` is the recommended pairing here, the scheduler settings above otherwise don't do much for it.<p>
#@markdown Recommended args for AdaFactor: `scale_parameter=False relative_step=False warmup_init=False`<p>
#@markdown If Dadapt or Prodigy (or ProdigyPlusScheduleFree) are selected and the recommended box is checked, the following values will override any previous settings:<p>
#@markdown `unet_lr=0.75` (`1.0` for ProdigyPlusScheduleFree specifically), `text_encoder_lr` to match, `network_alpha=network_dim`, `full_precision=False`<p>
recommended_values = True #@param {type:"boolean"}
#@markdown Alternatively set your own optimizer arguments separated by spaces (not commas). Recommended box must be disabled.
optimizer_args = "" #@param {type:"string"}
optimizer_args = [a.strip() for a in optimizer_args.split(' ') if a]

# sd-scripts doesn't know the friendly display name above -- it needs the actual importable
# module.Class path so it can dynamically import prodigy-plus-schedule-free at runtime.
OPTIMIZER_TYPE_MAP = {"ProdigyPlusScheduleFree": "prodigyplus.ProdigyPlusScheduleFree"}
optimizer_type_arg = OPTIMIZER_TYPE_MAP.get(optimizer, optimizer)

if recommended_values:
  if any(opt in optimizer.lower() for opt in ["dadapt", "prodigy"]):
    unet_lr = 1.0 if optimizer == "ProdigyPlusScheduleFree" else 0.75
    text_encoder_lr = unet_lr
    network_alpha = network_dim
    full_precision = False
  if optimizer == "Prodigy":
    optimizer_args = ["decouple=True", "weight_decay=0.01", "betas=[0.9,0.999]", "d_coef=2", "use_bias_correction=True", "safeguard_warmup=True"]
  elif optimizer == "ProdigyPlusScheduleFree":
    optimizer_args = ["weight_decay=0.1"]
  elif optimizer == "AdamW8bit":
    optimizer_args = ["weight_decay=0.1", "betas=[0.9,0.99]"]
  elif optimizer == "AdaFactor":
    optimizer_args = ["scale_parameter=False", "relative_step=False", "warmup_init=False"]

lr_scheduler_num_cycles = lr_scheduler_number
lr_scheduler_power = lr_scheduler_number

# Misc
seed = 42
bucket_reso_steps = 64
min_bucket_reso = 256
max_bucket_reso = 4096


#@markdown ### ▶️ Ready
#@markdown You can now run this cell to cook your Lora. Good luck! <p>


# 👩‍💻 Cool code goes here

root_dir = "/content"
trainer_dir = os.path.join(root_dir, "trainer")
kohya_dir = os.path.join(trainer_dir, "sd_scripts")

venv_python = os.path.join(kohya_dir, "venv/bin/python")
venv_pip = os.path.join(kohya_dir, "venv/bin/pip")
venv_accelerate = os.path.join(kohya_dir, "venv/bin/accelerate")
train_network_script = os.path.join(kohya_dir, "anima_train_network.py")

if "/Loras" in folder_structure:
  main_dir      = os.path.join(root_dir, "drive/MyDrive/Loras")
  log_folder    = os.path.join(main_dir, "_logs")
  config_folder = os.path.join(main_dir, project_name)
  images_folder = os.path.join(main_dir, project_name, "dataset")
  output_folder = os.path.join(main_dir, project_name, "output")
else:
  main_dir      = os.path.join(root_dir, "drive/MyDrive/lora_training")
  images_folder = os.path.join(main_dir, "datasets", project_name)
  output_folder = os.path.join(main_dir, "output", project_name)
  config_folder = os.path.join(main_dir, "config", project_name)
  log_folder    = os.path.join(main_dir, "log")

config_file = os.path.join(config_folder, "training_config.toml")
dataset_config_file = os.path.join(config_folder, "dataset_config.toml")


def install_xformers():
  # Only called if cross_attention is actually set to "xformers" -- most people leave it on the
  # default "torch" (sdpa), so we don't force everyone to pay for this download. Pinned to the
  # exact build that matches torch 2.6.0 + cu124: an unpinned `pip install xformers` can silently
  # grab a wheel built for a different torch/CUDA combo, which is what causes
  # memory_efficient_attention to come back as None at runtime.
  !{venv_pip} install xformers==0.0.29.post3 --index-url https://download.pytorch.org/whl/cu124 -q
  xformers_ok = os.system(f'{venv_python} -c "import xformers.ops" > /dev/null 2>&1') == 0
  if not xformers_ok:
    print("⚠️ xformers failed to import after installation. Switch cross_attention to \"torch\" instead, or the trainer will crash once it starts.")
  return xformers_ok


def install_trainer():
  !apt -y update -qq
  !apt install -y python3.10-venv aria2 -qq

  # Fetch only the single pinned release tag instead of a full clone -- much less to download,
  # and re-running this later will always get the exact same code, same as a full clone + reset
  # would, since we're asking for one specific immutable tag rather than "whatever main is now".
  os.makedirs(kohya_dir, exist_ok=True)
  os.chdir(kohya_dir)
  !git init -q
  !git remote add origin https://github.com/kohya-ss/sd-scripts
  !git fetch --depth 1 origin {SDSCRIPTS_TAG} -q
  !git checkout -q FETCH_HEAD

  !python3.10 -m venv venv
  !{venv_pip} install --upgrade pip -q
  !{venv_pip} install torch==2.6.0 torchvision==0.21.0 --index-url https://download.pytorch.org/whl/cu124 -q
  !{venv_pip} install --upgrade -r requirements.txt -q
  if cross_attention == "xformers":
    install_xformers()

  # fix logging
  !{venv_pip} uninstall -y rich -q

  if LOAD_TRUNCATED_IMAGES:
    !sed -i 's/from PIL import Image/from PIL import Image, ImageFile\nImageFile.LOAD_TRUNCATED_IMAGES=True/g' library/train_util.py # fix truncated jpegs error
  if BETTER_EPOCH_NAMES:
    !sed -i 's/{:06d}/{:02d}/g' library/train_util.py # make epoch names shorter
    !sed -i 's/"." + args.save_model_as)/"-{:02d}.".format(num_train_epochs) + args.save_model_as)/g' train_network.py # name of the last epoch will match the rest
  if wandb_key:
    !sed -i 's/accelerator.log(logs, step=epoch + 1)//g' train_network.py  # fix warning

  # non-interactive equivalent of `accelerate config`: single machine, single GPU, no distributed training
  !{venv_accelerate} config default --mixed_precision fp16

  os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
  os.environ["BITSANDBYTES_NOWELCOME"] = "1"
  os.environ["SAFETENSORS_FAST_GPU"] = "1"
  os.environ["PYTHONWARNINGS"] = "ignore"
  os.chdir(root_dir)


def validate_dataset():
  global lr_warmup_steps, lr_warmup_ratio, caption_extension, keep_tokens, model_url
  supported_types = (".png", ".jpg", ".jpeg", ".webp", ".bmp")

  if model_url.startswith("/content/drive/") and not os.path.exists(model_url):
    print("💥 Error: The custom training model you specified was not found in your Google Drive.")
    return

  print("\n💿 Checking dataset...")
  if not project_name.strip() or any(c in project_name for c in " .()\"'\\/"):
    print("💥 Error: Please choose a valid project name.")
    return

  # Find the folders and files
  if custom_dataset:
    try:
      datconf = toml.loads(custom_dataset)
      datasets = [d for d in datconf["datasets"][0]["subsets"]]
    except:
      print(f"💥 Error: Your custom dataset is invalid or contains an error! Please check the original template.")
      return
    reg = [d.get("image_dir") for d in datasets if d.get("is_reg", False)]
    datasets_dict = {d["image_dir"]: d["num_repeats"] for d in datasets}
    folders = datasets_dict.keys()
    files = [f for folder in folders for f in os.listdir(folder)]
    images_repeats = {folder: (len([f for f in os.listdir(folder) if f.lower().endswith(supported_types)]), datasets_dict[folder]) for folder in folders}
  else:
    reg = []
    folders = [images_folder]
    files = os.listdir(images_folder)
    images_repeats = {images_folder: (len([f for f in files if f.lower().endswith(supported_types)]), num_repeats)}

  # Validation
  for folder in folders:
    if not os.path.exists(folder):
      print(f"💥 Error: The folder {folder.replace('/content/drive/', '')} doesn't exist.")
      return
  for folder, (img, rep) in images_repeats.items():
    if not img:
      print(f"💥 Error: Your {folder.replace('/content/drive/', '')} folder is empty.")
      return
  test_files = []
  for f in files:
    if not f.lower().endswith((caption_extension, ".npz")) and not f.lower().endswith(supported_types):
      print(f"💥 Error: Invalid file in dataset: \"{f}\". Aborting.")
      return
    for ff in test_files:
      if f.endswith(supported_types) and ff.endswith(supported_types) \
          and os.path.splitext(f)[0] == os.path.splitext(ff)[0]:
        print(f"💥 Error: The files {f} and {ff} cannot have the same name. Aborting.")
        return
    test_files.append(f)

  if caption_extension and not [txt for txt in files if txt.lower().endswith(caption_extension)]:
    caption_extension = ""
  if continue_from_lora and not (continue_from_lora.endswith(".safetensors") and os.path.exists(continue_from_lora)):
    print(f"💥 Error: Invalid path to existing Lora. Example: /content/drive/MyDrive/Loras/example.safetensors")
    return

  # Show estimations to the user

  pre_steps_per_epoch = sum(img*rep for (img, rep) in images_repeats.values())
  steps_per_epoch = pre_steps_per_epoch/train_batch_size
  total_steps = max_train_steps or int(max_train_epochs*steps_per_epoch)
  estimated_epochs = int(total_steps/steps_per_epoch)
  lr_warmup_steps = int(total_steps*lr_warmup_ratio)

  for folder, (img, rep) in images_repeats.items():
    print("📁"+folder.replace("/content/drive/", "") + (" (Regularization)" if folder in reg else ""))
    print(f"📈 Found {img} images with {rep} repeats, equaling {img*rep} steps.")
  print(f"📉 Divide {pre_steps_per_epoch} steps by {train_batch_size} batch size to get {steps_per_epoch} steps per epoch.")
  if max_train_epochs:
    print(f"🔮 There will be {max_train_epochs} epochs, for around {total_steps} total training steps.")
  else:
    print(f"🔮 There will be {total_steps} steps, divided into {estimated_epochs} epochs and then some.")

  if total_steps > 10000:
    print("💥 Error: Your total steps are too high. You probably made a mistake. Aborting...")
    return

  return True


def create_config():
  global dataset_config_file, config_file, model_file

  if override_config_file:
    config_file = override_config_file
    print(f"\n⭕ Using custom config file {config_file}")
  else:
    config_dict = {
      "model_arguments": {
        "pretrained_model_name_or_path": model_file,
        "qwen3": qwen3_file,
        "vae": vae_file,
      },
      "network_arguments": {
        "unet_lr": unet_lr,
        "text_encoder_lr": text_encoder_lr if not cache_text_encoder_outputs else 0,
        "network_dim": network_dim,
        "network_alpha": network_alpha,
        "network_module": network_module,
        "network_args": network_args,
        "network_train_unet_only": text_encoder_lr == 0 or cache_text_encoder_outputs,
        "network_weights": continue_from_lora or None
      },
      "optimizer_arguments": {
        "learning_rate": unet_lr,
        "lr_scheduler": lr_scheduler,
        "lr_scheduler_num_cycles": lr_scheduler_num_cycles if lr_scheduler == "cosine_with_restarts" else None,
        "lr_scheduler_power": lr_scheduler_power if lr_scheduler == "polynomial" else None,
        "lr_warmup_steps": lr_warmup_steps if lr_scheduler not in ("cosine", "constant") else None,
        "optimizer_type": optimizer_type_arg,
        "optimizer_args": optimizer_args or None,
        "loss_type": "l2",
        "max_grad_norm": 1.0,
      },
      "training_arguments": {
        "lowram": LOWRAM,
        "max_train_steps": max_train_steps,
        "max_train_epochs": max_train_epochs,
        "train_batch_size": train_batch_size,
        "seed": seed,
        "attn_mode": "xformers" if cross_attention == "xformers" else "torch",
        "split_attn": cross_attention == "xformers",
        "no_half_vae": True,
        "gradient_checkpointing": True,
        "gradient_accumulation_steps": gradient_accumulation_steps,
        "max_data_loader_n_workers": 1,
        "persistent_data_loader_workers": True,
        "mixed_precision": mixed_precision,
        "full_fp16": mixed_precision == "fp16" and full_precision,
        "full_bf16": mixed_precision == "bf16" and full_precision,
        "cache_latents": cache_latents,
        "cache_latents_to_disk": cache_latents_to_drive,
        "cache_text_encoder_outputs": cache_text_encoder_outputs,
        "prior_loss_weight": 1.0,
        "timestep_sampling": timestep_sampling,
        "discrete_flow_shift": discrete_flow_shift,
        "sigmoid_scale": SIGMOID_SCALE,
        "weighting_scheme": weighting_scheme,
        "blocks_to_swap": blocks_to_swap or None,
        "vae_chunk_size": vae_chunk_size or None,
      },
      "saving_arguments": {
        "save_precision": "fp16",
        "save_model_as": "safetensors",
        "save_every_n_epochs": save_every_n_epochs,
        "save_last_n_epochs": keep_only_last_n_epochs,
        "output_name": project_name,
        "output_dir": output_folder,
        "log_prefix": project_name,
        "logging_dir": log_folder,
        "wandb_api_key": wandb_key or None,
        "log_with": "wandb" if wandb_key else None,
      }
    }

    for key in config_dict:
      if isinstance(config_dict[key], dict):
        config_dict[key] = {k: v for k, v in config_dict[key].items() if v is not None}

    with open(config_file, "w") as f:
      f.write(toml.dumps(config_dict))
    print(f"\n📄 Config saved to {config_file}")

  if override_dataset_config_file:
    dataset_config_file = override_dataset_config_file
    print(f"⭕ Using custom dataset config file {dataset_config_file}")
  else:
    dataset_config_dict = {
      "general": {
        "resolution": resolution,
        "shuffle_caption": shuffle_caption and not cache_text_encoder_outputs,
        "keep_tokens": keep_tokens,
        "flip_aug": False,
        "caption_extension": caption_extension,
        "enable_bucket": True,
        "bucket_no_upscale": False,
        "bucket_reso_steps": bucket_reso_steps,
        "min_bucket_reso": min_bucket_reso,
        "max_bucket_reso": max_bucket_reso,
      },
      "datasets": toml.loads(custom_dataset)["datasets"] if custom_dataset else [
        {
          "subsets": [
            {
              "num_repeats": num_repeats,
              "image_dir": images_folder,
              "class_tokens": None if caption_extension else project_name
            }
          ]
        }
      ]
    }

    for key in dataset_config_dict:
      if isinstance(dataset_config_dict[key], dict):
        dataset_config_dict[key] = {k: v for k, v in dataset_config_dict[key].items() if v is not None}

    with open(dataset_config_file, "w") as f:
      f.write(toml.dumps(dataset_config_dict))
    print(f"📄 Dataset config saved to {dataset_config_file}")


def download_model():
  global old_model_url, model_url, model_file
  real_model_url = model_url

  # Text encoder and VAE are fixed regardless of which diffusion model is used
  if not os.path.exists(qwen3_file):
    print("🔄 Downloading Qwen3 text encoder...")
    !aria2c "{qwen3_url}" --console-log-level=warn -c -s 16 -x 16 -k 10M -d / -o "{qwen3_file}"
  if not os.path.exists(vae_file):
    print("🔄 Downloading Qwen-Image VAE...")
    !aria2c "{vae_url}" --console-log-level=warn -c -s 16 -x 16 -k 10M -d / -o "{vae_file}"

  if real_model_url.startswith("/content/drive/"):
    # Local model, already checked to exist
    model_file = real_model_url
    print(f"📁 Using local model file: {model_file}")

  else:
    # Downloadable model
    if not real_model_url.lower().endswith(".safetensors"):
      if m := re.search(r"(?:https?://)?(?:www\.)?huggingface\.co/[^/]+/[^/]+/blob", real_model_url):
        real_model_url = real_model_url.replace("blob", "resolve")
      elif m := re.search(r"(?:https?://)?(?:www\.)?civitai\.com/models/([0-9]+)(/[A-Za-z0-9-_]+)?", real_model_url):
        if m.group(2):
          model_file = f"/content{m.group(2)}.safetensors"
        if mv := re.search(r"modelVersionId=([0-9]+)", real_model_url):
          real_model_url = f"https://civitai.com/api/download/models/{mv.group(1)}"
        else:
          raise ValueError("💥 optional_custom_training_model contains a civitai link, but the link doesn't include a modelVersionId. You can also right click the download button to copy the direct download link.")
      else:
        raise ValueError("💥 optional_custom_training_model must be a direct link to a single .safetensors file (this is how Anima and its finetunes are normally published, e.g. under split_files/diffusion_models on Hugging Face), a civitai model link, or a path in your Google Drive. A Diffusers-format repo like Anima-Base-v1.0-Diffusers can't be used here since this trainer needs the single-file checkpoint.")

    # Define local filename
    if not model_file or (old_model_url and old_model_url != model_url):
      if real_model_url.lower().endswith(".safetensors"):
        model_file = f"/content{real_model_url[real_model_url.rfind('/'):]}"
      else:
        model_file = "/content/downloaded_model.safetensors"
        if os.path.exists(model_file):
          !rm "{model_file}"

    # Download checkpoint
    print("🔄 Downloading Anima diffusion model...")
    !aria2c "{real_model_url}" --console-log-level=warn -c -s 16 -x 16 -k 10M -d / -o "{model_file}"

  # Validation
  from safetensors.torch import load_file as load_safetensors
  try:
    test = load_safetensors(model_file)
    del test
  except Exception as e:
    print(f"💥 Could not load {model_file} as a safetensors file ({e}).")
    return False

  return True


def run_streaming(cmd, cwd=None, env=None):
  """Run cmd and stream its combined stdout+stderr live into this cell's own output.

  We deliberately do NOT use `!command` (shell magic) here. tqdm's step progress bar updates a
  single line using bare \\r characters with no trailing newline. Colab's `!` shell-out relays a
  child subprocess's output back to the cell, but in practice it can sit on those \\r-only updates
  and only flush them in bursts (that's why epoch/checkpoint messages -- which end in a real
  newline -- show up fine, while the live steps: NN%|... bar doesn't). Reading the child's output
  ourselves and re-printing it via this cell's own stdout sidesteps that: prints made directly by
  the notebook's own kernel process render \\r updates properly, same as any live progress bar
  you've seen running directly in a Colab cell.
  """
  process = subprocess.Popen(
      cmd, cwd=cwd, env=env,
      stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=0,
  )
  decoder = codecs.getincrementaldecoder("utf-8")(errors="replace")
  try:
    while True:
      chunk = process.stdout.read(1024)
      if chunk:
        sys.stdout.write(decoder.decode(chunk))
        sys.stdout.flush()
      elif process.poll() is not None:
        break
    sys.stdout.write(decoder.decode(b"", final=True))
    sys.stdout.flush()
  except KeyboardInterrupt:
    process.terminate()
    raise
  return process.wait()


def main():
  global dependencies_installed

  if not os.path.exists('/content/drive'):
    from google.colab import drive
    print("📂 Connecting to Google Drive...")
    drive.mount('/content/drive')

  for dir in (main_dir, trainer_dir, log_folder, images_folder, output_folder, config_folder):
    os.makedirs(dir, exist_ok=True)

  if not validate_dataset():
    return

  if not dependencies_installed:
    print("\n🏭 Installing trainer...\n")
    t0 = time()
    install_trainer()
    t1 = time()
    dependencies_installed = True
    print(f"\n✅ Installation finished in {int(t1-t0)} seconds.")
  else:
    print("\n✅ Dependencies already installed.")

  if old_model_url != model_url or not model_file or not os.path.exists(model_file):
    print("\n🔄 Getting model...")
    if not download_model():
      print("\n💥 Error: The model you specified is invalid or corrupted."
            "\nIf you're using an URL, please check that the model is accessible without being logged in."
            "\nYou can try civitai or huggingface URLs, or a path in your Google Drive starting with /content/drive/MyDrive")
      return
    print()
  else:
    print("\n🔄 Model already downloaded.\n")

  if cross_attention == "xformers":
    xformers_ok = os.system(f'{venv_python} -c "import xformers.ops" > /dev/null 2>&1') == 0
    if not xformers_ok:
      print("\n🔄 Installing xformers (only needed the first time you select it)...")
      install_xformers()

  create_config()

  print("\n⭐ Starting trainer...\n")

  run_env = os.environ.copy()
  run_env["PYTHONUNBUFFERED"] = "1"
  exit_code = run_streaming(
      [venv_python, "-u", train_network_script,
       f"--config_file={config_file}", f"--dataset_config={dataset_config_file}"],
      cwd=kohya_dir, env=run_env,
  )

  if exit_code == 0:
    display(Markdown("### ✅ Done! [Go download your Lora from Google Drive](https://drive.google.com/drive/my-drive)\n"
                     "### There will be several files, you should try the latest version (the file with the largest number next to it)"))
  else:
    print(f"\n💥 The trainer exited with an error (code {exit_code}). Scroll up for the full log.")

main()

## *️⃣ Extras

You can run these before starting the training.

### 📚 Multiple folders in dataset
Below is a template allowing you to define multiple folders in your dataset. You must include the location of each folder and you can set different number of repeats for each one. To add more folders simply copy and paste the sections starting with `[[datasets.subsets]]`.

When enabling this, the number of repeats set in the main cell will be ignored, and the main folder set by the project name will also be ignored.

You can make one of them a regularization folder by adding `is_reg = true`  
You can also set different `keep_tokens`, `flip_aug`, etc.

In [ ]:
custom_dataset = """
[[datasets]]

[[datasets.subsets]]
image_dir = "/content/drive/MyDrive/Loras/example/dataset/good_images"
num_repeats = 3

[[datasets.subsets]]
image_dir = "/content/drive/MyDrive/Loras/example/dataset/normal_images"
num_repeats = 1

"""

In [ ]:
custom_dataset = None

In [ ]:
#@markdown ### 📂 Unzip dataset
#@markdown It's much slower to upload individual files to your Drive, so you may want to upload a zip if you have your dataset in your computer.
zip = "/content/drive/MyDrive/my_dataset.zip" #@param {type:"string"}
extract_to = "/content/drive/MyDrive/Loras/example/dataset" #@param {type:"string"}

import os, zipfile

if not os.path.exists('/content/drive'):
  from google.colab import drive
  print("📂 Connecting to Google Drive...")
  drive.mount('/content/drive')

os.makedirs(extract_to, exist_ok=True)

with zipfile.ZipFile(zip, 'r') as f:
  f.extractall(extract_to)

print("✅ Done")


In [ ]:
#@markdown ### 🔢 Count datasets
#@markdown Google Drive makes it impossible to count the files in a folder, so this will show you the file counts in all folders and subfolders.
folder = "/content/drive/MyDrive/Loras" #@param {type:"string"}

import os
from google.colab import drive

if not os.path.exists('/content/drive'):
    print("📂 Connecting to Google Drive...\n")
    drive.mount('/content/drive')

tree = {}
exclude = ("_logs", "/output")
for i, (root, dirs, files) in enumerate(os.walk(folder, topdown=True)):
  dirs[:] = [d for d in dirs if all(ex not in d for ex in exclude)]
  images = len([f for f in files if f.lower().endswith((".png", ".jpg", ".jpeg"))])
  captions = len([f for f in files if f.lower().endswith(".txt")])
  others = len(files) - images - captions
  path = root[folder.rfind("/")+1:]
  tree[path] = None if not images else f"{images:>4} images | {captions:>4} captions |"
  if tree[path] and others:
    tree[path] += f" {others:>4} other files"

pad = max(len(k) for k in tree)
print("\n".join(f"📁{k.ljust(pad)} | {v}" for k, v in tree.items() if v))


In [ ]:
#@markdown ### ↪️ Continue

#@markdown Here you can write a path in your Google Drive to load an existing Lora file to continue training on.<p>
#@markdown **Warning:** It's not the same as one long training session. The epochs start from scratch, and it may have worse results.
continue_from_lora = "" #@param {type:"string"}
if continue_from_lora and not continue_from_lora.startswith("/content/drive/MyDrive"):
  import os
  continue_from_lora = os.path.join("/content/drive/MyDrive", continue_from_lora)


# 📈 Plot training results
You can do this after running the trainer. You don't need this unless you know what you're doing.  
The first cell below may fail to load all your logs. Keep trying the second cell until all data has loaded.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir={log_folder}/

In [ ]:
from tensorboard import notebook
notebook.display(port=6006, height=800)